# SignBridge M3 — Word-Sign GRU Model Training

This notebook downloads a public ASL word-sign dataset (WLASL), extracts MediaPipe hand + pose landmarks, trains a GRU temporal model, and exports it to TensorFlow.js for the browser.

**Expected runtime:** ~1–3 hours on Colab GPU (T4)
**Output:** `gru-word-signs/` folder saved to your Google Drive — ready to copy into `web/public/models/`

---
### What this notebook does
1. Installs dependencies & clones the SignBridge repo
2. Downloads WLASL videos for the 25 target word signs
3. Extracts MediaPipe landmarks (159-dim: both hands + upper-body pose)
4. Trains a 2-layer GRU classifier (~300K params)
5. Exports to TF.js LayersModel format
6. Saves everything to your Google Drive

### Prerequisites
- Google Drive with ~3 GB free space (videos + artifacts)
- GPU runtime: **Runtime → Change runtime type → T4 GPU**

In [ ]:
# ── Cell 1: Install dependencies ───────────────────────────────────────
!pip install -q mediapipe==0.10.35 opencv-python>=4.10.0 tqdm scikit-learn
!pip install -q tensorflow>=2.16.0 tensorflowjs>=4.22.0
!pip install -q yt-dlp

import os, json, sys, subprocess, shutil
from collections import Counter
from pathlib import Path

print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/signbridge-m3'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Artifacts will be saved to: {DRIVE_OUTPUT}")

In [ ]:
# ── Cell 3: Clone SignBridge repo ──────────────────────────────────────
REPO_URL = 'https://github.com/mhmdtaha091/SignBridge.git'
REPO_DIR = '/content/SignBridge'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}/ml
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Cell 4: Apply MediaPipe API typo fixes (safety net) ────────────────
# The two typos below cause runtime crashes. Patch them in case the repo
# hasn't been updated yet.

PREPROCESS_PATH = 'preprocess_temporal.py'
with open(PREPROCESS_PATH, 'r') as f:
    src = f.read()

src = src.replace('min_hand_presence_confience', 'min_hand_presence_confidence')
src = src.replace('min_pose_presence_confience', 'min_pose_presence_confidence')

with open(PREPROCESS_PATH, 'w') as f:
    f.write(src)

print("✅ Patched preprocess_temporal.py (MediaPipe API typos)")

In [ ]:
# ── Cell 5: Load vocabulary map ─────────────────────────────────────────
VOCAB_MAP_PATH = os.path.join('colab', 'vocab_map.json')

with open(VOCAB_MAP_PATH) as f:
    GLOSS_TO_WORD = json.load(f)

TARGET_WORDS = sorted(set(GLOSS_TO_WORD.values()))
print(f"Target words: {len(TARGET_WORDS)}")
print(f"WLASL glosses to match: {len(GLOSS_TO_WORD)}")
print()
for i, w in enumerate(TARGET_WORDS):
    glosses = [g for g, wid in GLOSS_TO_WORD.items() if wid == w]
    print(f"  {i:2d}. {w:20s} ← {', '.join(glosses)}")

In [ ]:
# ── Cell 6: Clone WLASL repo & download videos ─────────────────────────
# We clone the official WLASL repo for the metadata JSON (WLASL_v0.3.json),
# then use yt-dlp to download available YouTube videos for our target words.

WLASL_REPO = '/content/WLASL'
if not os.path.exists(WLASL_REPO):
    !git clone https://github.com/dxli94/WLASL.git {WLASL_REPO}

# Load WLASL metadata.
with open(os.path.join(WLASL_REPO, 'start_kit', 'WLASL_v0.3.json')) as f:
    wlasl_data = json.load(f)

print(f"Total WLASL entries: {len(wlasl_data)}")

# Filter to entries whose gloss matches our vocab map.
target_entries = []
for entry in wlasl_data:
    gloss = entry['gloss'].strip().upper()
    if gloss in GLOSS_TO_WORD:
        target_entries.append(entry)

print(f"Matching entries: {len(target_entries)}")

# Count per word.
per_word = Counter(GLOSS_TO_WORD[e['gloss'].strip().upper()] for e in target_entries)
print("\nPer-word availability in WLASL metadata:")
for word, count in per_word.most_common():
    print(f"  {word:20s}  {count:4d} videos")

missing = [w for w in TARGET_WORDS if w not in per_word]
if missing:
    print(f"\n⚠ Words NOT found in WLASL: {missing}")
    print("  These will need hand-recorded data later.")

In [ ]:
# ── Cell 7: Download available videos ───────────────────────────────────
import urllib.request
import time
import random

VIDEO_DIR = os.path.join('data', 'wlasl_videos')
LABELS_CSV_PATH = os.path.join('data', 'wlasl_labels.csv')
os.makedirs(VIDEO_DIR, exist_ok=True)

downloaded = 0
failed = 0
label_rows = []
per_word_dl = Counter()

# Only process entries where we don't already have the video.
for i, entry in enumerate(target_entries):
    gloss = entry['gloss'].strip().upper()
    word_id = GLOSS_TO_WORD[gloss]
    video_id = entry['video_id']
    url = entry['url']

    # Create per-word subdirectory.
    word_dir = os.path.join(VIDEO_DIR, word_id)
    os.makedirs(word_dir, exist_ok=True)

    out_path = os.path.join(word_dir, f"{video_id}.mp4")
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        # Already downloaded.
        label_rows.append((f"{word_id}/{video_id}.mp4", word_id))
        downloaded += 1
        per_word_dl[word_id] += 1
        continue

    # Download with yt-dlp.
    try:
        result = subprocess.run(
            ['yt-dlp', '-q', '--no-warnings', '-f', 'mp4', '-o', out_path,
             '--max-filesize', '50M', url],
            capture_output=True, text=True, timeout=60
        )
        if result.returncode == 0 and os.path.exists(out_path):
            label_rows.append((f"{word_id}/{video_id}.mp4", word_id))
            downloaded += 1
            per_word_dl[word_id] += 1
        else:
            failed += 1
    except Exception as e:
        failed += 1

    # Progress.
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(target_entries)}  ok={downloaded} fail={failed}")

    # Be polite to YouTube.
    time.sleep(random.uniform(0.3, 0.8))

# Write labels CSV for preprocess_temporal.py.
with open(LABELS_CSV_PATH, 'w') as f:
    for filename, label in label_rows:
        f.write(f"{filename},{label}\n")

print(f"\nDownloaded: {downloaded}  Failed: {failed}")
print(f"Labels CSV: {LABELS_CSV_PATH}")
print("\nPer-word downloaded:")
for word, count in per_word_dl.most_common():
    print(f"  {word:20s}  {count:4d} videos")

still_missing = [w for w in TARGET_WORDS if per_word_dl.get(w, 0) == 0]
if still_missing:
    print(f"\n⚠ No downloadable videos for: {still_missing}")
    print("  Hand-record these signs later using Data Studio in the SignBridge app.")

if downloaded < 30:
    print(f"\n⚠ Only {downloaded} videos total — model quality will be poor.")
    print("  Consider hand-recording additional samples.")

In [ ]:
# ── Cell 8: Verify GPU ──────────────────────────────────────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU available: {gpus}")
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else:
    print("⚠ No GPU detected. Training will work but be slower.")
    print("  Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Cell 9: Preprocess — extract MediaPipe landmarks from videos ────────
# Runs preprocess_temporal.py on the downloaded videos.
# Each video frame → MediaPipe Hands (both hands) + Pose → 159-dim features
# → segmented into 60-frame sliding windows.

PREPROCESS_CMD = [
    sys.executable, 'preprocess_temporal.py',
    '--input_dir', 'data/wlasl_videos/',
    '--labels', 'data/wlasl_labels.csv',
    '--output', 'data/temporal',
    '--window_size', '60',
    '--stride', '15',
    '--every_n_frames', '1',
]

print("Running:", ' '.join(PREPROCESS_CMD))
print()
result = subprocess.run(PREPROCESS_CMD)
if result.returncode != 0:
    print(f"\n❌ Preprocessing failed (exit code {result.returncode})")
    sys.exit(1)
print("\n✅ Preprocessing complete")

In [ ]:
# ── Cell 10: Inspect preprocessed data ──────────────────────────────────
import numpy as np

NPZ_PATH = os.path.join('data', 'temporal', 'temporal_landmarks.npz')
LABELS_PATH = os.path.join('data', 'temporal', 'temporal_labels.json')

if not os.path.exists(NPZ_PATH):
    print(f"❌ {NPZ_PATH} not found. Preprocessing may have failed.")
    sys.exit(1)

X = np.load(NPZ_PATH)['X']
with open(LABELS_PATH) as f:
    labels_raw = json.load(f)

unique_labels = sorted(set(labels_raw))
label_counts = Counter(labels_raw)

print(f"Total windows:  {X.shape[0]:,}")
print(f"Window shape:   {X.shape[1:]}  (frames={X.shape[1]}, features={X.shape[2]})")
print(f"Classes:        {len(unique_labels)}")
print(f"Memory:         {X.nbytes / 1024**2:.1f} MB")
print()
print("Class distribution:")
for label, count in label_counts.most_common():
    bar = '█' * min(50, count // max(1, label_counts.most_common(1)[0][1] // 50))
    print(f"  {label:20s} {count:5d}  {bar}")

if len(unique_labels) < 5:
    print(f"\n⚠ Only {len(unique_labels)} classes — model quality will be limited.")

# Check for severe imbalance.
counts = list(label_counts.values())
if len(counts) > 1 and max(counts) / min(counts) > 20:
    print("\n⚠ Severe class imbalance detected (>20:1). Class weights will be used.")

In [ ]:
# ── Cell 11: Train the GRU model ────────────────────────────────────────
# Architecture: GRU(128) → Dropout(0.3) → GRU(64) → Dropout(0.3) → Softmax
# ~300K params, ~1.2 MB when exported to TF.js

TRAIN_CMD = [
    sys.executable, 'train_temporal.py',
    '--data_dir', 'data/temporal',
    '--output_dir', 'models',
    '--epochs', '80',
    '--batch_size', '32',
    '--lr', '1e-3',
    '--val_split', '0.15',
]

print("Running:", ' '.join(TRAIN_CMD))
print()
result = subprocess.run(TRAIN_CMD)
if result.returncode != 0:
    print(f"\n❌ Training failed (exit code {result.returncode})")
    sys.exit(1)
print("\n✅ Training complete")

In [ ]:
# ── Cell 12: Load & report final metrics ────────────────────────────────
LABELS_OUT = os.path.join('models', 'labels.json')
H5_PATH = os.path.join('models', 'signbridge_gru.h5')

if not os.path.exists(LABELS_OUT):
    print(f"❌ {LABELS_OUT} not found — training may have failed.")
    sys.exit(1)

with open(LABELS_OUT) as f:
    labels_meta = json.load(f)

print(f"Model saved: {H5_PATH}  ({os.path.getsize(H5_PATH) / 1024**2:.1f} MB)")
print(f"Vocabulary:  {len(labels_meta['labels'])} words")
print(f"Val accuracy: {labels_meta.get('val_accuracy', 'N/A')}")
print(f"Words: {labels_meta['labels']}")

In [ ]:
# ── Cell 13: Export to TensorFlow.js ────────────────────────────────────
# Converts the .h5 Keras model to TF.js LayersModel format that the browser
# can load via tf.loadLayersModel().

TFJS_OUT = 'tfjs_model/gru-word-signs'

EXPORT_CMD = [
    sys.executable, 'export_tfjs.py',
    '--model', H5_PATH,
    '--labels', LABELS_OUT,
    '--output', TFJS_OUT,
]

print("Running:", ' '.join(EXPORT_CMD))
print()
result = subprocess.run(EXPORT_CMD)
if result.returncode != 0:
    print(f"\n❌ Export failed (exit code {result.returncode})")
    sys.exit(1)
print("\n✅ TF.js export complete")

In [ ]:
# ── Cell 14: Verify TF.js model ─────────────────────────────────────────
MODEL_JSON = os.path.join(TFJS_OUT, 'model.json')
VOCAB_JSON = os.path.join(TFJS_OUT, 'vocab.json')

assert os.path.exists(MODEL_JSON), f"❌ {MODEL_JSON} missing!"
assert os.path.exists(VOCAB_JSON), f"❌ {VOCAB_JSON} missing!"

with open(MODEL_JSON) as f:
    model_meta = json.load(f)
with open(VOCAB_JSON) as f:
    vocab_meta = json.load(f)

total_size = sum(
    os.path.getsize(os.path.join(TFJS_OUT, f))
    for f in os.listdir(TFJS_OUT)
)

print(f"✅ TF.js model verified")
print(f"   model.json:  {os.path.getsize(MODEL_JSON):,} bytes")
print(f"   total size:  {total_size / 1024:.1f} KB")
print(f"   format:      {model_meta.get('format', 'unknown')}")
print(f"   labels:      {vocab_meta.get('labels', [])}")
print(f"   val acc:     {vocab_meta.get('val_accuracy', 'N/A')}")
print()
print("Output files:")
for f in sorted(os.listdir(TFJS_OUT)):
    size = os.path.getsize(os.path.join(TFJS_OUT, f))
    print(f"   {f:40s} {size:>8,} bytes")

In [ ]:
# ── Cell 15: Save to Google Drive ───────────────────────────────────────
DEST = os.path.join(DRIVE_OUTPUT, 'gru-word-signs')

if os.path.exists(DEST):
    shutil.rmtree(DEST)
shutil.copytree(TFJS_OUT, DEST)

# Write a human-readable summary.
from datetime import datetime
summary_path = os.path.join(DEST, 'training_summary.txt')
with open(summary_path, 'w') as f:
    f.write("SignBridge M3 — GRU Word-Sign Model\n")
    f.write(f"Trained: {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}\n")
    f.write(f"Dataset: WLASL (dxli94/WLASL, YouTube videos)\n")
    f.write(f"Architecture: GRU(128)→Dropout(0.3)→GRU(64)→Dropout(0.3)→Softmax\n")
    f.write(f"Input: 60 frames × 159 features (both hands + upper-body pose)\n")
    f.write(f"Classes: {vocab_meta.get('labels', [])}\n")
    f.write(f"Validation accuracy: {vocab_meta.get('val_accuracy', 'N/A')}\n")
    f.write(f"Model size: {total_size / 1024:.1f} KB\n")

print(f"✅ Saved to Google Drive: {DEST}")
print()
print("Files:")
for f in sorted(os.listdir(DEST)):
    print(f"   {f}")

---
## Next Steps

### 1. Download the model from Google Drive
The `gru-word-signs/` folder is at `MyDrive/signbridge-m3/gru-word-signs/` in your Google Drive.
Download it to your local machine.

### 2. Copy into the SignBridge web app
Place the folder at:
```
SignBridge/web/public/models/gru-word-signs/
├── model.json
├── group1-shard1of1.bin
└── vocab.json
```

### 3. Verify label alignment
The next cell prints the exact label order the model expects. Make sure your `web/src/config/vocab.ts` WORD_SIGNS array contains these same words (order doesn't need to match — the model loads labels from `vocab.json`).

### 4. Deploy
```bash
cd SignBridge/web
vercel --prod
```

### 5. Hand-record missing signs
If any of the 25 target words had 0 videos in WLASL, use the SignBridge Data Studio to record them, then re-run training with the combined dataset.

In [ ]:
# ── Cell 16: Print model label order (for vocab.ts verification) ────────
print("Model output label order (index → label):")
print()
for i, label in enumerate(vocab_meta['labels']):
    print(f"  {i:2d}: '{label}'")

print()
print("Paste this into your browser console to verify against WORD_SIGNS:")
print(f"  const modelLabels = {json.dumps(vocab_meta['labels'])};")
print(f"  const vocabWords = {json.dumps(TARGET_WORDS)};")
print("  console.log('In model, not in vocab:', modelLabels.filter(w => !vocabWords.includes(w)));")
print("  console.log('In vocab, not in model:', vocabWords.filter(w => !modelLabels.includes(w)));")

# Check set difference.
model_set = set(vocab_meta['labels'])
vocab_set = set(TARGET_WORDS)
in_model_not_vocab = model_set - vocab_set
in_vocab_not_model = vocab_set - model_set

if in_model_not_vocab:
    print(f"\n⚠ Words in model but NOT in vocab.ts: {sorted(in_model_not_vocab)}")
if in_vocab_not_model:
    print(f"\n⚠ Words in vocab.ts but NOT in model: {sorted(in_vocab_not_model)}")
    print("  Hand-record these and re-train, or accept they won't be recognized.")
if not in_model_not_vocab and not in_vocab_not_model:
    print("\n✅ All 25 target words are in the trained model — perfect match!")

---
## Troubleshooting

| Problem | Solution |
|---|---|
| **No GPU** | Runtime → Change runtime type → T4 GPU, then re-run Cell 8 |
| **yt-dlp "URL not found"** | YouTube video was removed. Normal — the notebook skips dead links. Check per-word counts in Cell 7 output. |
| **"No hand frames in video"** | Video has no clear hand visible. Normal — the preprocessing script skips these. |
| **preprocess_temporal.py crashes** | Re-run Cell 4 (typo patch), then re-run Cell 9 |
| **tensorflowjs_converter not found** | Re-run Cell 1 (`pip install tensorflowjs`) |
| **Colab disconnects (idle timeout)** | Re-run from last completed cell. The `ModelCheckpoint` callback saves the best model, so you won't lose training progress. |
| **Drive quota exceeded** | Free up Google Drive space, then re-run Cell 15 |
| **Very low val accuracy (<30%)** | Not enough training data. Try hand-recording additional samples per word. Or check that the downloaded videos actually show the correct sign. |
| **Model labels don't match vocab.ts** | Some WLASL glosses may have been missed. Add more gloss mappings to `vocab_map.json` in the repo, push, and re-run. |